# RAG from a Live News Web Page — built entirely with **LangChain**

**Classroom demo.** An LLM's knowledge is frozen at training time, so it may not know
about recent events. **Retrieval-Augmented Generation (RAG)** fixes this at *query time*:
we fetch relevant text from an external source (here, a news web page), put it into the
prompt, and let the LLM answer from that supplied context.

**Example event:** the *glacier collapse and flooding in Nepal, August 2026*.

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` (small, fast, runs on the free Colab T4 GPU).

This notebook uses LangChain for **every** step — nothing is hand-rolled:

| Stage | LangChain piece |
|-------|-----------------|
| Load the web page      | `WebBaseLoader` → list of `Document` |
| Split into chunks      | `RecursiveCharacterTextSplitter` |
| Turn text into vectors | `HuggingFaceEmbeddings` |
| Store & search vectors | `FAISS` vector store |
| Fetch relevant chunks  | `vectorstore.as_retriever()` |
| Prompt template        | `ChatPromptTemplate` |
| The LLM                | `HuggingFacePipeline` + `ChatHuggingFace` |
| Wire it together       | LCEL — the `|` pipe operator + `RunnablePassthrough.assign` |

> **Before running:** `Runtime → Change runtime type → Hardware accelerator → T4 GPU`,
> then run the cells top to bottom.

## 1. Install the required libraries

We only need the LangChain packages plus the model/embedding backends.
`transformers`, `accelerate`, `requests` and `beautifulsoup4` already ship with Colab,
so we don't reinstall them (that also avoids the harmless `google-colab` version
warning).

In [ ]:
# langchain-core           -> Runnables / LCEL (the | operator), prompts, output parsers
#                             (pulled in automatically by the packages below)
# langchain-community      -> WebBaseLoader, FAISS vector store wrapper
# langchain-huggingface    -> HuggingFacePipeline, ChatHuggingFace, HuggingFaceEmbeddings
# langchain-text-splitters -> RecursiveCharacterTextSplitter
# faiss-cpu                -> the actual vector index
# sentence-transformers    -> the embedding model backend
#
# We do NOT need the big `langchain` meta-package: this notebook builds its chain
# from langchain-core primitives only, which keeps the install small and avoids the
# version conflicts that package sometimes causes on Colab.
#
# "requests==2.32.4" is pinned to the version Colab ships, so pip does not upgrade it
# and you don't get the "google-colab 1.0.0 requires requests==2.32.4" conflict warning.
!pip install -q "requests==2.32.4" langchain-core langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers beautifulsoup4

# show what actually got installed (helps debug import errors)
import importlib.metadata as _md
for _p in ["langchain-core", "langchain-community", "langchain-huggingface",
           "langchain-text-splitters", "faiss-cpu", "sentence-transformers", "transformers"]:
    try:
        print(f"{_p:24s} {_md.version(_p)}")
    except Exception:
        print(f"{_p:24s} NOT INSTALLED")

print("\nInstall finished. If Colab shows a 'restart runtime' button, click it, then re-run from here.")

## 2. Check the Colab environment & make output readable

Shows the hardware, and sets up **line wrapping** so long answers wrap inside the
cell instead of running off the screen (important when projecting in class).

In [ ]:
# --- make long output lines WRAP instead of scrolling off the screen ---------------
# (1) CSS: force streamed text output to wrap inside the Colab cell
from IPython.display import HTML, display
display(HTML(
    "<style>"
    "pre, .output pre, .jp-RenderedText pre, .output-plaintext "
    "{ white-space: pre-wrap !important; word-break: break-word; }"
    "</style>"
))

# (2) helper: print a (possibly long) string wrapped to a fixed column width
import textwrap
def show(text, width=100):
    for line in (str(text).splitlines() or [""]):
        print(textwrap.fill(line, width=width, replace_whitespace=False) if line else "")

# (3) quiet transformers' harmless per-call notices (e.g. "Both max_new_tokens and
#     max_length seem to have been set"). Comment out to see all warnings.
import transformers
transformers.logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")


import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU name        :", torch.cuda.get_device_name(0))
    print(f"GPU memory      : {props.total_memory / 1024**3:.1f} GB")
    # Compute capability 7.5 = Turing (the T4). Turing has fast fp16 but no bf16.
    print(f"Compute capab.  : {props.major}.{props.minor}")
else:
    print("No GPU detected. Everything still runs on CPU, just slower.")
    print("Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

# nvidia-smi prints the full GPU status table (driver, memory, running processes)
!nvidia-smi || echo "(no GPU runtime)"

## 3. The LangChain building blocks (quick reference)

Keep these seven objects in mind — the whole notebook is just plugging them together:

1. **`Document`** — a piece of text plus a `metadata` dict (`page_content`, `metadata`).
2. **Document loader** (`WebBaseLoader`, `PyPDFLoader`, …) — produces `Document`s from a source.
3. **Text splitter** (`RecursiveCharacterTextSplitter`) — cuts long `Document`s into chunks.
4. **Embeddings** (`HuggingFaceEmbeddings`) — `text -> list[float]` vector.
5. **Vector store** (`FAISS`) — stores chunk vectors and does similarity search.
6. **Retriever** — a thin wrapper over the vector store: `question -> top-k Documents`.
7. **Runnable / LCEL** — every LangChain component is a *Runnable*; the `|` operator
   pipes the output of one into the next, e.g. `prompt | llm | parser`.

A **chain** is just Runnables composed together with `|`. Two helpers we lean on:

* **`RunnablePassthrough`** — passes its input through unchanged.
* **`RunnablePassthrough.assign(key=runnable)`** — runs `runnable` on the current dict
  and *adds* the result back under `key`. Chaining several `.assign(...)` steps lets us
  keep every intermediate value (retrieved docs, formatted context, final answer) so we
  can print them — which is exactly what a classroom demo needs.

## 4. Load `Qwen2.5-1.5B-Instruct` as a LangChain chat model

Two LangChain wrappers are involved:

* **`HuggingFacePipeline`** — wraps a 🤗 `transformers` text-generation pipeline so
  LangChain can call it. All decoding settings (temperature, top_p, …) live here.
* **`ChatHuggingFace`** — adds the *chat* interface on top: it takes a list of
  `system` / `human` / `ai` messages and applies Qwen2.5's chat template automatically.

**Why fp16 and no quantization?** 1.5B parameters in fp16 ≈ 3 GB — it fits the T4
easily next to the embedding model, and fp16 matrix-multiply is fast on the T4.
4-bit quantization would only add complexity here.

**Why `Qwen2.5-*-Instruct`?** It is a plain instruction model: no hidden "thinking"
mode, no `<think>…</think>` blocks — the reply *is* the final answer.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, set_seed
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # swap to "Qwen/Qwen2.5-3B-Instruct" for stronger answers

# --- load the weights + tokenizer once; we reuse these objects everywhere ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

use_gpu = torch.cuda.is_available()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if use_gpu else torch.float32,  # T4 -> fp16
    device_map="auto" if use_gpu else None,
)
base_model.eval()

# default generation_config has max_length=20; we always pass max_new_tokens, so clear
# max_length to silence the "Both max_new_tokens and max_length ... set" warning
base_model.generation_config.max_length = None

print("Model loaded on:", next(base_model.parameters()).device)


# Return a LangChain ChatHuggingFace with the given DECODING settings.
# Rebuilding is cheap because `base_model` / `tokenizer` are already in memory - we
# only wrap them in a light pipeline. This lets us make several chat models with
# different temperature/top_p/top_k to compare (see section 5).
def build_chat_model(max_new_tokens=300, do_sample=True, temperature=0.7,
                     top_p=0.9, top_k=50, repetition_penalty=1.1):
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,          # hard cap on how many tokens to generate
        do_sample=do_sample,                    # False = greedy (always pick the top token)
        repetition_penalty=repetition_penalty,  # >1.0 discourages repeating the same words
        return_full_text=False,                 # return only the new text, not the prompt
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    if do_sample:
        # these three only matter when sampling is on
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    text_gen_pipeline = pipeline(
        task="text-generation",
        model=base_model,
        tokenizer=tokenizer,
        **gen_kwargs,
    )
    # drop the default max_length=20 so it doesn't clash with max_new_tokens
    text_gen_pipeline.model.generation_config.max_length = None
    hf_llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
    return ChatHuggingFace(llm=hf_llm, tokenizer=tokenizer)


# a general-purpose chat model for the "no RAG" and creativity demos
chat_model = build_chat_model(temperature=0.7, top_p=0.9, top_k=50)

# quick sanity check
from langchain_core.messages import HumanMessage
print(chat_model.invoke([HumanMessage(content="Reply with exactly: RAG demo ready.")]).content)

## 5. What do `temperature`, `top_p`, `top_k`, … actually do?

At each step the model produces a **probability for every token** in its vocabulary.
The decoding parameters decide *how* the next token is picked from that distribution.

| Parameter | Meaning | Effect |
|-----------|---------|--------|
| `do_sample=False` (greedy) | always take the single highest-probability token | 100% deterministic, can feel flat / repetitive |
| `do_sample=True` | pick a token *at random*, weighted by probability | varied output; controlled by the knobs below |
| `temperature` | divides the logits before softmax. `<1` sharpens the distribution (safer, more predictable); `=1` leaves it as-is; `>1` flattens it (more surprising / "creative"); `→0` ≈ greedy | higher = more random |
| `top_k` | before sampling, keep only the `k` most-likely tokens | small `k` = safer vocabulary, less rambling |
| `top_p` (nucleus) | keep the smallest set of tokens whose probabilities add up to `p` (e.g. 0.9) | adapts: few tokens when the model is confident, more when it's unsure |
| `repetition_penalty` | down-weights tokens that already appeared | `>1` reduces loops like "the the the" |
| `max_new_tokens` | maximum number of tokens to generate | length / cost / speed cap |

**Rule of thumb**
* *Creative writing:* `temperature ≈ 0.9–1.2`, `top_p ≈ 0.95`.
* *Factual / RAG answers:* low `temperature ≈ 0.0–0.3` so the model sticks to the
  retrieved facts and gives repeatable answers.

Let's see it. We use one deliberately open-ended prompt and change only the knobs.

In [ ]:
CREATIVE_PROMPT = "In one vivid sentence, describe a river flowing through a mountain valley."

def sample_twice(chat, label, seed_a=1, seed_b=2):
    # set_seed makes a sampled run reproducible; two different seeds show the SPREAD
    set_seed(seed_a); out_a = chat.invoke(CREATIVE_PROMPT).content.strip()
    set_seed(seed_b); out_b = chat.invoke(CREATIVE_PROMPT).content.strip()
    print("=" * 90)
    print(label)
    print("-" * 90)
    show("run 1: " + out_a)
    show("run 2: " + out_b)

short = dict(max_new_tokens=45)

# 1) GREEDY: no randomness at all -> both runs are identical
sample_twice(build_chat_model(do_sample=False, **short),
             "GREEDY  (do_sample=False)  -> deterministic, identical every time")

# 2) LOW temperature: sampling on, but very conservative -> runs are close
sample_twice(build_chat_model(temperature=0.2, top_p=1.0, top_k=0, **short),
             "LOW temperature = 0.2      -> minor variation, plays it safe")

# 3) HIGH temperature: distribution flattened -> runs diverge, wording gets bolder
sample_twice(build_chat_model(temperature=1.3, top_p=1.0, top_k=0, **short),
             "HIGH temperature = 1.3     -> wild, very different each run")

# 4) top_k = 5: only ever choose among the 5 likeliest tokens -> safe vocabulary
sample_twice(build_chat_model(temperature=1.0, top_k=5, top_p=1.0, **short),
             "top_k = 5                  -> creativity capped to the 5 best tokens")

# 5) top_p = 0.5: only the most probable tokens summing to 50% are eligible
sample_twice(build_chat_model(temperature=1.0, top_p=0.5, top_k=0, **short),
             "top_p = 0.5 (nucleus)      -> tight nucleus, focused wording")

**Takeaway for RAG:** in the sections below the answer model uses a **low temperature**
so that, given the same retrieved chunks, it produces the same grounded answer every
time and does not "improvise" beyond the article.

## 6. Ask the LLM **without** RAG

We ask about the recent event with **no article supplied** — pure pretrained knowledge.

In [ ]:
QUESTION = "What happened in the recent glacier collapse in Nepal in August 2026?"

# plain chat call: one human message in, one AI message out
no_rag_reply = chat_model.invoke(QUESTION).content

print("QUESTION:", QUESTION)
print("\n--- answer from pretrained knowledge only ---\n")
show(no_rag_reply.strip())

The model answers from whatever it saw during training. For a very recent event the
answer may be vague, incomplete, or partly guessed. We are **not** claiming it knows
nothing — we just observe the limitation: **its knowledge has a cut-off date.**

## 7. Load the news web page with `WebBaseLoader`

`WebBaseLoader` downloads the URL and (via BeautifulSoup) turns the HTML into a
LangChain `Document`. We pass a `SoupStrainer` so BeautifulSoup **only parses the
`<p>` / heading tags** — that automatically drops `<script>`, `<style>`, nav bars,
menus and cookie banners, which live in other tags.

`NEWS_URL` is a real Al Jazeera article. If Colab can't fetch it (some news sites
block datacentre IPs), the code falls back to the Wikipedia article on the same
event so the notebook always runs end-to-end.

In [ ]:
import os
# WebBaseLoader warns if this isn't set; a descriptive UA is polite to servers
os.environ["USER_AGENT"] = "Mozilla/5.0 (classroom-rag-demo; educational use)"

import bs4
from langchain_community.document_loaders import WebBaseLoader

NEWS_URL     = "https://www.aljazeera.com/news/2026/8/27/nepal-tibet-floods-what-happened-what-caused-them-and-who-is-missing"
FALLBACK_URL = "https://en.wikipedia.org/wiki/2026_Nepal_floods"

# only keep paragraph + heading text when parsing the HTML
article_strainer = bs4.SoupStrainer(["p", "h1", "h2", "h3", "li"])

def load_web_article(url):
    loader = WebBaseLoader(
        web_paths=[url],
        bs_kwargs={"parse_only": article_strainer},
        requests_kwargs={"timeout": 30},
    )
    docs = loader.load()                       # -> list[Document] (one per URL)
    # collapse runs of blank lines that the strainer leaves behind
    for d in docs:
        d.page_content = "\n".join(line.strip() for line in d.page_content.splitlines() if line.strip())
    return docs

# try the news site first, fall back to Wikipedia on error or thin content
try:
    web_docs = load_web_article(NEWS_URL)
    if len(web_docs[0].page_content.split()) < 200:
        raise ValueError("page returned too little text (site may be blocking us)")
    SOURCE_URL = NEWS_URL
except Exception as e:
    print("Primary URL unusable:", e, "\n-> falling back to Wikipedia\n")
    web_docs = load_web_article(FALLBACK_URL)
    SOURCE_URL = FALLBACK_URL

print("Source used :", SOURCE_URL)
print("Documents   :", len(web_docs))
print("Word count  :", len(web_docs[0].page_content.split()))
print("Metadata    :", web_docs[0].metadata)
print("\n---------------- TEXT PREVIEW (first 1200 chars) ----------------\n")
show(web_docs[0].page_content[:1200])

## 8. Split the article into chunks with `RecursiveCharacterTextSplitter`

**Why chunk?**
* The retriever should return *small, focused* passages, not the whole article.
* Long context is slower, costlier, and dilutes the relevant sentences.

`RecursiveCharacterTextSplitter` tries to split on paragraph breaks first, then
sentences, then words — so chunks stay semantically clean. `chunk_overlap` repeats a
little text across the boundary so a fact split between two chunks still appears whole
in at least one of them.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,        # ~ a few paragraphs (characters, not words)
    chunk_overlap=200,      # carry 200 chars into the next chunk
    separators=["\n\n", "\n", ". ", " ", ""],   # try these split points in order
)

chunks = splitter.split_documents(web_docs)     # list[Document] -> more, smaller Documents

print("Number of chunks:", len(chunks))
print("Each chunk keeps the source metadata:", chunks[0].metadata)
print("\n---------------- SAMPLE CHUNK [1] ----------------\n")
show(chunks[1].page_content)

## 9. Create embeddings with `HuggingFaceEmbeddings`

An **embedding** maps a piece of text to a fixed-length vector so that texts with
similar meaning land close together in vector space. That is what lets us retrieve by
*meaning* instead of keyword matching.

Model: `sentence-transformers/all-MiniLM-L6-v2` — tiny (~90 MB), fast, 384-dim output,
runs fine on the free Colab GPU or CPU. We L2-normalise the vectors so that
"closeness" is effectively **cosine similarity**.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},   # -> cosine-similarity friendly
)

# peek at what an embedding looks like
demo_vec = embeddings.embed_query("glacier collapse in Nepal")
print("Embedding dimension:", len(demo_vec))
print("First 8 numbers    :", [round(x, 4) for x in demo_vec[:8]])

## 10. Build a FAISS vector store

**FAISS** (Facebook AI Similarity Search) is a local library that stores many vectors
and searches them quickly. No server, no cloud, no API key — the index lives in Colab's
RAM. `FAISS.from_documents` embeds every chunk and adds it to the index in one call.

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)   # embed + index every chunk

print("Vectors stored in the FAISS index:", vectorstore.index.ntotal)

# (optional) you could persist it with vectorstore.save_local("faiss_nepal")

## 11. Turn the vector store into a retriever

`as_retriever()` gives a component whose job is simply **question → top-k Documents**.
We also run `similarity_search_with_score` directly so students can *see* the distance
scores (with FAISS these are **L2 distances**: smaller = more similar).

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})   # return the 3 closest chunks

probe = "What caused the disaster?"

print("QUESTION:", probe)
print("\n--- retrieved chunks + FAISS L2 distance (lower = closer) ---\n")
for rank, (doc, score) in enumerate(vectorstore.similarity_search_with_score(probe, k=3), start=1):
    print(f"[#{rank}] distance={score:.3f}")
    show(doc.page_content[:300].strip() + " ...")
    print()

## 12. Write the RAG prompt with `ChatPromptTemplate`

The prompt has two slots that LangChain fills in automatically:

* `{context}` — the retrieved chunks (LangChain concatenates the `Document`s here)
* `{input}`   — the student's question

The system message pins the model to the supplied context.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_RULES = (
    "You are a factual assistant for a classroom demo. "
    "Answer the question using ONLY the context provided below. "
    "If the answer is not in the context, reply exactly: "
    "'The information is not available in the provided article.' "
    "Be concise and do not add outside knowledge."
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULES),
    ("human", "Context from the news article:\n\n{context}\n\nQuestion: {input}"),
])

print(rag_prompt.format(context="<retrieved chunks go here>", input="<question goes here>"))

## 13. Assemble the RAG chain (pure LCEL)

We build the chain from `langchain-core` primitives only — no extra `langchain`
package needed. Each `RunnablePassthrough.assign(...)` step **adds a key** to the dict
travelling through the chain:

```
{"input": question}
   │  .assign(docs    = retrieve top-k chunks for input)
   ▼
{"input", "docs"}
   │  .assign(context = format the docs into one text block)
   ▼
{"input", "docs", "context"}
   │  .assign(answer  = rag_prompt | answer_model | StrOutputParser)
   ▼
{"input", "docs", "context", "answer"}
```

Because every intermediate value survives, `ask_rag` can print the retrieved chunks
*and* the answer. The answer model uses **temperature 0.1** — we want grounded,
repeatable answers, not creative writing.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# low temperature -> the model sticks to the retrieved facts and repeats itself run-to-run
answer_model = build_chat_model(temperature=0.1, top_p=0.9, top_k=20, max_new_tokens=300)

def format_docs(docs):
    # turn the list[Document] from the retriever into one plain-text block for {context}
    return "\n\n".join(f"[chunk {i}] {d.page_content}" for i, d in enumerate(docs, start=1))

# the sub-chain that actually calls the LLM: fill the prompt -> generate -> plain string.
# it receives the whole dict; ChatPromptTemplate just picks out {context} and {input}.
generate_answer = rag_prompt | answer_model | StrOutputParser()

# the full RAG chain, composed with the | operator
rag_chain = (
    RunnablePassthrough.assign(docs=lambda x: retriever.invoke(x["input"]))     # 1. retrieve
    | RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))      # 2. format
    | RunnablePassthrough.assign(answer=generate_answer)                        # 3. answer
)

# helper that prints the whole pipeline transparently: question -> chunks -> answer
def ask_rag(question, seed=42, show_chunks=True):
    set_seed(seed)                                   # reproducible generation
    result = rag_chain.invoke({"input": question})   # -> dict with input/docs/context/answer
    print("QUESTION:", question)
    if show_chunks:
        print("\n-- retrieved chunks (exactly what the LLM was given) --")
        for i, d in enumerate(result["docs"], start=1):
            show(f"[chunk {i}] {d.page_content[:220].strip()} ...")
    print("\n-- grounded answer --")
    show(result["answer"].strip())
    print()
    return result

_ = ask_rag("What caused the disaster?")

## 14. Run RAG on several questions

For each one: **Question → Retrieved chunks → Grounded answer.**

In [ ]:
for q in [
    "What caused the disaster?",
    "Which areas were affected?",
    "What happened after the glacier collapse?",
    "What risks remain?",
    "What are the authorities doing?",
]:
    print("#" * 95)
    ask_rag(q)

## 15. Side by side: **without RAG** vs **with RAG**

Same question, two conditions.

In [ ]:
compare_q = "What happened in the recent glacier collapse in Nepal in August 2026?"

print("=" * 95)
print("A)  LLM ALONE  (no retrieval)")
print("=" * 95)
set_seed(42)
show(chat_model.invoke(compare_q).content.strip())

print("\n" + "=" * 95)
print("B)  LLM + RAG  (same question + retrieved article chunks)")
print("=" * 95)
set_seed(42)
show(rag_chain.invoke({"input": compare_q})["answer"].strip())

RAG did **not** retrain the model. `answer_model`'s weights are identical before and
after. All that changed is that the retrieved chunks were placed into the prompt at
inference time. Ask B again in a fresh notebook with an empty vector store and you get
answer A back.

## 16. The minimal chain — when you only need the answer

Section 13 kept every intermediate value so we could print the chunks. If you just
want the answer string, the whole RAG pipeline is four pipes. The first step is a
dict: LangChain runs both entries on the input question (a plain string) and builds
`{"context": ..., "input": ...}` for the prompt.

```
{"context": retriever | format_docs,   "input": passthrough}
        | rag_prompt         # fill the template
        | answer_model       # call the LLM
        | StrOutputParser()  # AIMessage -> plain string
```

In [ ]:
# reuses format_docs, rag_prompt and answer_model from section 13
minimal_rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | rag_prompt
    | answer_model
    | StrOutputParser()
)

set_seed(42)
show(minimal_rag_chain.invoke("Which areas were affected?").strip())

## 17. Summary

**Traditional LLM**

```
Question ──► LLM ──► Answer
```

**RAG (this notebook)**

```
Question
   │
   ▼
Retriever  ── embed the question, search FAISS ──►  top-k chunks from the web page
   │
   ▼
Prompt  (system rules + chunks + question)
   │
   ▼
LLM  (Qwen2.5-1.5B-Instruct, low temperature)
   │
   ▼
Grounded answer
```

**RAG does not update the model's weights.** The external document is retrieved and
supplied as context *at inference time*. Remove the retrieval step and the model is
back to its pretrained knowledge.

### Things to try next

* **Ingest multiple pages** — pass several URLs to `WebBaseLoader` (or call it in a
  loop) and add all chunks to the same FAISS store.
* **Retrieve from PDFs** — swap `WebBaseLoader` for `PyPDFLoader`; nothing else changes.
* **Add metadata** — tag each chunk with source URL / date / section and print it with
  every answer for citations.
* **Tune chunking** — change `chunk_size` / `chunk_overlap` and watch retrieval quality.
* **Tune `k`** — `search_kwargs={"k": ...}`; more context isn't always better.
* **Try other embedding models** — `BAAI/bge-small-en-v1.5`, `thenlper/gte-small`, …
* **Add a reranker** — `ContextualCompressionRetriever` + a `CrossEncoderReranker`
  after the FAISS step.
* **Tune decoding** — revisit section 5; raise `answer_model`'s temperature and see the
  answers drift away from the article.
* **Use a persistent / local vector DB** — `Chroma`, or `FAISS.save_local` +
  `FAISS.load_local`.
* **Swap the LLM** — any HF chat model, or a hosted one via `ChatOpenAI` /
  `ChatGoogleGenerativeAI` (needs an API key).